# Experimentos complementarios — LightGBM y mezclas v2
**Equipo 20 · Proyecto ML–SEDP · Cuaderno independiente de `Cierre_correcciones`**

Este notebook concentra **los demás experimentos ya comprobados** con LightGBM, sus mezclas con el Voting corregido y sus métricas. Su propósito es **comparar opciones**, no modificar el modelo provisional adoptado en el cierre.

**Alcance:** misma tabla `FE_TOP_Model_Table_v2.csv`, mismo esquema espacial del Avance 5, correcciones B+C, preparadores e interacciones de referencia v2. Incluye LightGBM con PCA, LightGBM raw ponderado, pérdidas asimétricas y ensamble triple; **no** agrega características v3, validación anidada, intervalos, ablaciones ni procesos nuevos pendientes de ejecutar.

Las métricas incluidas como **referencia histórica** proceden de la ejecución previa de `Experimentos_TDPA_LightGBM_v3`. Las celdas de código permiten reproducir las recetas v2 cuando estén disponibles los módulos y los CSV originales; **no se han ejecutado aquí**.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics.pairwise import haversine_distances
from threadpoolctl import threadpool_limits

import tdpa_comun as tc
from tdpa_comun import TARGET, PreparadorSeguro, preparar_datos
from validacion_regiones import construir_folds_regiones

%matplotlib inline
warnings.filterwarnings("ignore")
SEED, THREADS = 20, 2
threadpool_limits(limits=THREADS)
np.random.seed(SEED)

ROOT = tc.encontrar_raiz()
TABLA = ROOT / "03_PROCESSED" / "FE_TOP_Model_Table_v2.csv"
TABLA_CTN = ROOT / "03_PROCESSED" / "FE_CTN_Prediction_Table_v2.csv"
OUT = ROOT / "06_RESULTS" / "EXPERIMENTOS_MODELOS_V2"
OUT.mkdir(parents=True, exist_ok=True)

tabla, train, test = preparar_datos(TABLA)
folds = list(construir_folds_regiones(train))
y = train[TARGET]
quintil_real = pd.qcut(train["tdpa"], 5, labels=["Q1 (bajo)", "Q2", "Q3", "Q4", "Q5 (alto)"])
CORTES = train["tdpa"].quantile([1/3, 2/3]).to_numpy()

## 0. Modelo base y evaluación reproducible

El Voting corregido (Ridge, Elastic Net, SVR y Gradient Boosting) utiliza **B+C+A**, con `fuerza=0.5`. Su predicción fuera de pliegue es la referencia para comparar todas las recetas. `resumen()` conserva las métricas del cierre original.

In [2]:
# Hiperparámetros elegidos en el Avance 5
ALPHA = 20
GB_MEJOR = {"learning_rate": 0.03, "max_iter": 150, "min_samples_leaf": 20, "l2_regularization": 0.0}

def pipe(estimador):
    return Pipeline([("prep", PreparadorSeguro(modo="Filtro + PCA")), ("modelo", estimador)])

def modelos_base():
    return [pipe(Ridge(alpha=ALPHA)),
            pipe(ElasticNet(alpha=0.01, l1_ratio=0.9, max_iter=20000, random_state=SEED)),
            pipe(SVR(kernel="rbf", C=10, epsilon=0.15, gamma="scale")),
            pipe(HistGradientBoostingRegressor(early_stopping=False, random_state=SEED, **GB_MEJOR))]

class VotingPonderado(BaseEstimator, RegressorMixin):
    # Promedio de los 4 modelos. fuerza > 0: peso = 1 + fuerza * (desviaciones estándar por encima de la mediana)
    def __init__(self, fuerza=0.0):
        self.fuerza = fuerza
    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        w = 1 + self.fuerza * np.clip((y - np.median(y)) / y.std(), 0, None)
        w = w / w.mean()
        self.modelos_ = [m.fit(X, y, modelo__sample_weight=w) for m in modelos_base()]
        return self
    def predict(self, X):
        return np.mean([m.predict(X) for m in self.modelos_], axis=0)

def predecir_cv(datos, fuerza=0.0):
    # Predicciones fuera de pliegue con la validación espacial del Avance 5
    oof = pd.Series(np.nan, index=datos.index)
    for _, itr, iva in folds:
        tr = datos.loc[itr]
        oof[iva] = VotingPonderado(fuerza).fit(tr, tr[TARGET]).predict(datos.loc[iva])
    return oof

def resumen(p):
    r = y - p
    alto, bajo = quintil_real == "Q5 (alto)", quintil_real == "Q1 (bajo)"
    pred = np.exp(p)
    return {"rmse_log": np.mean([np.sqrt(((y[iva] - p[iva]) ** 2).mean()) for _, _, iva in folds]),
            "spearman": spearmanr(y, p)[0],
            "acierto_categoria": float(np.mean(np.digitize(train["tdpa"], CORTES) == np.digitize(pred, CORTES))),
            "sesgo_Q5_alto": r[alto].mean(), "rmse_Q5_alto": np.sqrt((r[alto] ** 2).mean()),
            "sesgo_Q1_bajo": r[bajo].mean(), "rmse_Q1_bajo": np.sqrt((r[bajo] ** 2).mean()),
            "wape_pct": 100 * np.abs(train["tdpa"] - pred).sum() / train["tdpa"].sum()}

OOF = {"Avance 5 (sin cambios)": predecir_cv(train)}
p0 = OOF["Avance 5 (sin cambios)"]
print({k: round(v, 4) for k, v in resumen(p0).items()})

{'rmse_log': np.float64(0.6744), 'spearman': np.float64(0.8129), 'acierto_categoria': 0.7055, 'sesgo_Q5_alto': np.float64(0.5033), 'rmse_Q5_alto': np.float64(0.7897), 'sesgo_Q1_bajo': np.float64(-0.6625), 'rmse_Q1_bajo': np.float64(0.927), 'wape_pct': np.float64(48.3055)}


## 1. Correcciones v2 comunes a todos los candidatos

Se corrige la cuota según SICT (B) y se repara el máximo flujo dentro de 25 km por carretera (C). La ponderación del Voting (A=0.5) y la de LightGBM Raw (`fuerza=3`) se calculan exclusivamente con el conjunto de entrenamiento de cada fold; **no significan lo mismo**.

In [ ]:
RADIO_KM = 25.0

def reparar_flujo(puntos, grupos, radio_km=RADIO_KM):
    out = puntos["red_flujo_paso"].copy()
    for _, g in puntos.groupby(grupos):
        if len(g) > 1:
            dist = haversine_distances(np.radians(g[["lat", "lon"]].to_numpy(float))) * 6371.0088
            v = g["red_flujo_paso"].to_numpy(float)
            out.loc[g.index] = [v[dist[i] <= radio_km].max() for i in range(len(g))]
    return out

flujo_reparado = reparar_flujo(tabla, tabla["carretera"])
def corregir(datos, cuota=True, flujo=True):
    d = datos.copy()
    if cuota:
        d["fe_peaje"] = np.maximum(d["fe_peaje"], d["toll_road"].astype(int))
    if flujo:
        d["red_flujo_paso"] = flujo_reparado.loc[d.index]
    return d

train_bc = corregir(train, cuota=True, flujo=True)

## 2. Preparadores de referencia v2 (sin características v3)

Se recupera la implementación verificada: preparador con PCA e interacciones v2, y preparador Raw sin PCA con las mismas cinco interacciones v2. El modo Raw sigue las variables aprobadas por `PreparadorSeguro`, excluyendo variables objetivo y auxiliares. Todas las transformaciones aprendidas se ajustan únicamente con el entrenamiento del fold.

In [ ]:
# Preparadores de la referencia v2. NO se importan ni crean las características v3.
from sklearn.base import TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy import sparse
from lightgbm import LGBMRegressor

def interacciones_v2(X):
    def numerica(columna):
        if columna not in X.columns:
            return np.zeros(len(X), dtype=float)
        return (pd.to_numeric(X[columna], errors="coerce")
                .replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(float))
    flujo = np.maximum(numerica("red_flujo_paso"), 0)
    peaje = np.clip(numerica("fe_peaje"), 0, 1)
    carriles = np.maximum(numerica("fe_carriles"), 0)
    return np.column_stack([
        np.log1p(flujo),
        np.log1p(flujo) * peaje,
        np.log1p(flujo) * np.log1p(carriles),
        peaje * np.log1p(carriles),
        (flujo <= 0).astype(float),
    ])

class PreparadorConInteraccionesV2(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.base_ = PreparadorSeguro(modo="Filtro + PCA").fit(X, y)
        self.escalador_ = StandardScaler().fit(interacciones_v2(X))
        return self
    def transform(self, X):
        a = self.base_.transform(X)
        a = a.toarray() if sparse.issparse(a) else np.asarray(a, dtype=float)
        return np.hstack([a, self.escalador_.transform(interacciones_v2(X))])

class PreparadorSinPCAV2(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        selector = PreparadorSeguro(modo="Filtro + PCA").fit(X, y)
        atributos = ("num_seleccionadas_", "cat_seleccionadas_", "pca_cols_")
        if not all(hasattr(selector, a) for a in atributos):
            raise AttributeError("PreparadorSeguro debe exponer las listas originales de variables v2")
        prohibidas = {str(TARGET).lower(), "tdpa", "lat", "lon", "km", "carretera", "ctn_id", "toll_road"}
        def admisible(c):
            s = str(c).lower()
            return s not in prohibidas and not any(
                k in s for k in ("tdpa", "aadt", "aforo", "prediccion", "target", "residuo"))
        self.cat_ = list(dict.fromkeys(c for c in selector.cat_seleccionadas_ if admisible(c)))
        self.num_ = list(dict.fromkeys(c for c in
                          [*selector.num_seleccionadas_, *selector.pca_cols_]
                          if admisible(c) and c not in self.cat_))
        if not (self.num_ or self.cat_):
            raise ValueError("No quedaron variables permitidas en el preparador sin PCA")
        if self.cat_:
            try:
                self.encoder_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
            except TypeError:
                self.encoder_ = OneHotEncoder(handle_unknown="ignore", sparse=False)
            self.encoder_.fit(self._categoricas(X))
        return self
    def _categoricas(self, X):
        return X[self.cat_].astype("string").fillna("__MISSING__").astype(str)
    def transform(self, X):
        faltantes = sorted(set(self.num_ + self.cat_) - set(X.columns))
        if faltantes:
            raise KeyError(f"Faltan variables para la referencia raw v2: {faltantes}")
        partes = []
        if self.num_:
            partes.append(X[self.num_].apply(pd.to_numeric, errors="coerce")
                          .replace([np.inf, -np.inf], np.nan).to_numpy(float))
        if self.cat_:
            partes.append(self.encoder_.transform(self._categoricas(X)))
        partes.append(interacciones_v2(X))
        return np.hstack(partes)

print("Preparadores v2 disponibles: PCA + 5 interacciones históricas / raw + 5 interacciones históricas")

## 3. Modelos LightGBM y recetas comparativas

Las configuraciones ya comprobadas combinan PCA o Raw, pérdida cuadrática o asimétrica, y ponderación del tránsito alto. Las mezclas se calculan **en escala log(TDPA)** y los coeficientes de Voting/LightGBM suman 1. Este espacio experimental se conserva separado del cierre principal.

In [ ]:
# Modelos LightGBM con la configuración ya probada en la referencia v2.
# El entrenamiento de cada fold calcula pesos y pérdida usando solamente sus propios aforos.
class LightGBMReferenciaV2(BaseEstimator, RegressorMixin):
    def __init__(self, preparacion="pca", perdida="l2", lam=1.0, fuerza=0.0):
        self.preparacion = preparacion
        self.perdida = perdida
        self.lam = lam
        self.fuerza = fuerza

    def fit(self, X, y):
        yy = np.asarray(y, dtype=float)
        self.prep_ = (PreparadorConInteraccionesV2() if self.preparacion == "pca"
                      else PreparadorSinPCAV2())
        z = self.prep_.fit_transform(X, yy)
        self.offset_ = float(np.median(yy)) if self.perdida == "asimetrica" else 0.0
        objetivo = "regression"
        if self.perdida == "asimetrica":
            centrado = yy - self.offset_
            q10, q50, q90 = np.quantile(centrado, [0.1, 0.5, 0.9])
            def objetivo(y_real, y_pred, weight):
                error = np.asarray(y_pred, float) - np.asarray(y_real, float)
                factor = np.where(error < 0, self.lam, 1.0)
                if weight is not None:
                    factor = factor * np.asarray(weight, dtype=float)
                return factor * error, factor
        elif self.perdida != "l2":
            raise ValueError("Pérdida no reconocida")
        self.modelo_ = LGBMRegressor(
            n_estimators=350, learning_rate=0.035, num_leaves=15, max_depth=5,
            min_child_samples=25, subsample=0.85, subsample_freq=0,
            colsample_bytree=0.8, reg_lambda=8, verbosity=-1,
            random_state=SEED, n_jobs=THREADS, objective=objetivo)
        peso = None
        if self.fuerza > 0:
            desviacion = max(float(np.std(yy)), 1e-8)
            peso = 1 + self.fuerza * np.maximum((yy - np.median(yy)) / desviacion, 0)
            peso = peso / peso.mean()
        self.modelo_.fit(z, yy - self.offset_, sample_weight=peso)
        return self

    def predict(self, X):
        p = np.asarray(self.modelo_.predict(self.prep_.transform(X)), dtype=float) + self.offset_
        if not np.isfinite(p).all():
            raise ValueError("LightGBM produjo predicciones no finitas")
        return p

CONFIG_LGB_V2 = {
    "pca_l2_a0": dict(preparacion="pca"),
    "raw_l2_a3": dict(preparacion="raw", fuerza=3),
    "raw_asym_l3": dict(preparacion="raw", perdida="asimetrica", lam=3),
    "pca_asym_l3": dict(preparacion="pca", perdida="asimetrica", lam=3),
}
# Las proporciones son coeficientes del promedio en log(TDPA), no pesos de muestra.
RECETAS_LGB_V2 = {
    "Voting corregido": {"Voting": 1.0},
    "pca_l2_a0 | LGB 60%": {"Voting": 0.4, "pca_l2_a0": 0.6},
    "raw_l2_a3 | LGB 80%": {"Voting": 0.2, "raw_l2_a3": 0.8},
    "raw_asym_l3 | LGB 80%": {"Voting": 0.2, "raw_asym_l3": 0.8},
    "pca_asym_l3 | LGB 80%": {"Voting": 0.2, "pca_asym_l3": 0.8},
    "pca_asym_l3 | LGB 100%": {"pca_asym_l3": 1.0},
    "Triple v2 | Voting 20%": {"Voting": 0.2, "raw_asym_l3": 0.4, "pca_asym_l3": 0.4},
}
assert all(abs(sum(v.values()) - 1) < 1e-10 for v in RECETAS_LGB_V2.values())

def ajustar_referencias_v2(datos):
    yy = datos[TARGET].astype(float)
    ajustados = {"Voting": VotingPonderado(fuerza=0.5).fit(datos, yy)}
    for clave, opciones in CONFIG_LGB_V2.items():
        ajustados[clave] = LightGBMReferenciaV2(**opciones).fit(datos, yy)
    return ajustados

def predecir_recetas_v2(modelos, datos):
    base = {clave: np.asarray(modelo.predict(datos), float) for clave, modelo in modelos.items()}
    return {nombre: sum(peso * base[clave] for clave, peso in receta.items())
            for nombre, receta in RECETAS_LGB_V2.items()}

def evaluar_cv_referencias_v2(datos):
    resultados = {nombre: pd.Series(np.nan, index=datos.index, dtype=float)
                  for nombre in RECETAS_LGB_V2}
    for numero, itr, iva in folds:
        ajustados = ajustar_referencias_v2(datos.loc[itr])
        predicciones = predecir_recetas_v2(ajustados, datos.loc[iva])
        for nombre, pred in predicciones.items():
            resultados[nombre].loc[iva] = pred
        print(f"Pliegue espacial {numero} completado: {len(itr)} entrenamiento, {len(iva)} validación")
    for nombre, pred in resultados.items():
        if pred.isna().any() or not np.isfinite(pred).all():
            raise ValueError(f"No se completaron las predicciones OOF de {nombre}")
    return resultados

### 3.1 Referencia histórica: resultados espaciales de las recetas v2

La tabla siguiente recoge los **resultados ya observados en `Experimentos_TDPA_LightGBM_v3` para las recetas sin nuevas variables v3**. Se muestra como **referencia histórica comprobada**, no como una nueva ejecución. La siguiente celda permite reproducirla sobre la tabla v2 original; necesita disponer del proyecto y de LightGBM.

| Modelo ya probado | RMSE log CV ↓ | RMSE Q5 ↓ | Sesgo Q5 ↓ | RMSE Q1 ↓ | WAPE ↓ |
|---|---:|---:|---:|---:|---:|
| Voting corregido B+C+A (0.5) | 0.6609 | 0.6791 | +0.4183 | 0.9273 | 48.74% |
| LightGBM PCA 60% + Voting 40% | **0.6332** | 0.6501 | +0.4435 | **0.8880** | 44.89% |
| LightGBM raw, ponderación 3, 80% + Voting 20% | 0.6375 | 0.6218 | +0.3894 | 0.9049 | **44.25%** |
| LightGBM raw, asimétrica λ=3, 80% + Voting 20% | 0.6425 | 0.6007 | +0.3571 | 0.9450 | 44.97% |
| LightGBM PCA, asimétrica λ=3, 80% + Voting 20% | 0.6437 | 0.5966 | +0.3541 | 0.9613 | 44.73% |
| LightGBM PCA, asimétrica λ=3, 100% | 0.6509 | **0.5930** | +0.3381 | 0.9769 | 44.77% |
| Triple v2: Voting 20% + LightGBM raw 40% + PCA 40% (asimétricos) | 0.6415 | 0.5972 | +0.3556 | 0.9519 | 44.68% |

**Lectura.** No hay un ganador único para todas las metas. La mezcla **PCA 60%** tiene el menor error global y mejor RMSE Q1; **raw ponderado 80%** tiene el WAPE más bajo y un mejor compromiso con el tránsito alto. La pérdida asimétrica y el triple bajan más el error Q5, pero empeoran Q1 respecto del Voting corregido. El resultado de **PCA asimétrico puro** es el mejor *sólo en Q5* de esta selección; no se adopta como ganador global. Son comparaciones OOF de recetas fijas ya examinadas sobre los mismos 703 sitios; no constituyen por sí mismas validación independiente de la elección.

In [ ]:
# Reproducción de los resultados ya comprobados: misma tabla, mismos folds, B+C y A=0.5.
# NO se agrega ingeniería de características v3 ni validación/procesos nuevos.
train_bc = corregir(train, cuota=True, flujo=True)
OOF_LGB_V2 = evaluar_cv_referencias_v2(train_bc)
comparacion_lgb_v2 = pd.DataFrame({nombre: resumen(pred) for nombre, pred in OOF_LGB_V2.items()}).T
comparacion_lgb_v2.to_csv(OUT / "comparacion_modelos_experimentales_v2.csv", encoding="utf-8-sig")
display(comparacion_lgb_v2.style.format({
    "rmse_log": "{:.4f}", "spearman": "{:.3f}", "acierto_categoria": "{:.1%}",
    "sesgo_Q5_alto": "{:+.3f}", "rmse_Q5_alto": "{:.3f}",
    "sesgo_Q1_bajo": "{:+.3f}", "rmse_Q1_bajo": "{:.3f}", "wape_pct": "{:.2f}"}))

# Auditoría: el Voting recompuesto debe coincidir con el Voting B+C+A del cierre.
rmse_voting_referencia = resumen(OOF_LGB_V2["Voting corregido"])["rmse_log"]
rmse_voting_cierre = resumen(predecir_cv(train_bc, fuerza=0.5))["rmse_log"]
assert np.isclose(rmse_voting_referencia, rmse_voting_cierre, atol=1e-5), (
    f"La réplica del Voting no coincide: {rmse_voting_referencia:.5f} vs {rmse_voting_cierre:.5f}")
print(f"Verificación del Voting B+C+A: RMSE log {rmse_voting_referencia:.4f}")

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
orden = comparacion_lgb_v2.sort_values("rmse_log", ascending=False).index
axs[0].barh(orden, comparacion_lgb_v2.loc[orden, "rmse_log"], color="tab:blue")
axs[0].axvline(rmse_voting_referencia, color="k", ls="--", lw=1, label="Voting corregido")
axs[0].set(title="Gráfico C3b. RMSE log, CV espacial", xlabel="RMSE log medio de 5 pliegues")
axs[0].legend(fontsize=8)
axs[1].barh(orden, comparacion_lgb_v2.loc[orden, "rmse_Q5_alto"], color="tab:orange", label="Q5 (alto)")
axs[1].barh(orden, comparacion_lgb_v2.loc[orden, "rmse_Q1_bajo"], color="tab:green", alpha=0.45, label="Q1 (bajo)")
axs[1].set(title="RMSE por extremo (log)", xlabel="RMSE log por quintil")
axs[1].set_yticklabels([])
axs[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "comparacion_modelos_experimentales_v2.png", dpi=120); plt.show()

## 4. Diagnóstico de los modelos en MEX-095D

En la ejecución histórica de referencia, el corredor mostró los siguientes RMSE log: **Voting corregido 0.6037**, **PCA 60% 0.6177**, **Raw ponderado 80% 0.5913**, **Raw asimétrico 80% 0.5750** y **triple asimétrico 0.5770**. Estos resultados **no determinan** la elección del modelo final porque el corredor ya se había consultado. La celda permite reproducir la tabla y una figura comparativa con los datos originales.

In [ ]:
T = test.sort_values("KM")
yT, km = T[TARGET].to_numpy(float), T["KM"].to_numpy(float)

def m_ext(p):
    r = yT - p
    return {"rmse_log": np.sqrt((r ** 2).mean()), "sesgo": r.mean(),
            "spearman": spearmanr(yT, p)[0],
            "acierto_categoria": float(np.mean(np.digitize(T["tdpa"], CORTES) == np.digitize(np.exp(p), CORTES)))}

# Diagnóstico externo fijo. Se entrena sólo en train; no se elige modelo con yT.
modelos_lgb_v2_final = ajustar_referencias_v2(train_bc)
test_bc = corregir(T, cuota=True, flujo=True)
externas_lgb_v2 = predecir_recetas_v2(modelos_lgb_v2_final, test_bc)
comparacion_externa_lgb_v2 = pd.DataFrame({nombre: m_ext(pred)
                                           for nombre, pred in externas_lgb_v2.items()}).T
comparacion_externa_lgb_v2.to_csv(OUT / "diagnostico_modelos_experimentales_v2.csv", encoding="utf-8-sig")
display(comparacion_externa_lgb_v2.style.format({
    "rmse_log": "{:.4f}", "sesgo": "{:+.3f}", "spearman": "{:.3f}",
    "acierto_categoria": "{:.0%}"}))

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(km, T["tdpa"], "o-", color="black", label="TDPA real (SICT)")
for nombre in ("Voting corregido", "pca_l2_a0 | LGB 60%",
               "raw_l2_a3 | LGB 80%", "raw_asym_l3 | LGB 80%",
               "Triple v2 | Voting 20%"):
    ax.plot(km, np.exp(externas_lgb_v2[nombre]), "--", lw=1.5, label=nombre)
ax.set_yscale("log")
ax.set(xlabel="kilómetro (SICT)", ylabel="TDPA (veh/día, escala log)",
       title="Gráfico C3c. Modelos ya probados en MEX-095D (diagnóstico)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig(OUT / "corredor_modelos_experimentales_v2.png", dpi=120); plt.show()

## 5. Lectura de los experimentos y límites

- La receta **PCA 60% + Voting 40%** obtuvo el menor RMSE global de las referencias mostradas (0.6332); la receta **Raw ponderada 80% + Voting 20%** obtuvo el menor WAPE de ese grupo (44.25%) y una mejora de Q5 respecto del Voting, manteniendo mejor control de Q1 que las alternativas asimétricas.
- Las recetas con **pérdida asimétrica** y el **triple** mejoraron más Q5, pero elevaron el error Q1. Por ello, una mejora localizada no equivale a un ganador universal.
- Las cifras provienen de **recetas fijas examinadas con los mismos pliegues espaciales** y de un corredor consultado previamente. No sustituyen una evaluación independiente después de seleccionar el modelo.
- El **cierre principal adopta provisionalmente sólo Raw ponderado 80% + Voting 20%**. Este notebook conserva la trazabilidad de las demás opciones; la futura ingeniería de características, validaciones nuevas y análisis experimentales v3 permanecen pendientes y **no se ejecutan aquí**.